# Literature Map

Visualize the paper collection: tag distributions, year trends, and citation analysis.

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from src.storage.paper_store import PaperStore

store = PaperStore()
papers = store.get_all()
print(f"Loaded {len(papers)} papers")

## Papers by Year

In [ ]:
years = [p.year for p in papers if p.year]
if years:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(years, bins=range(min(years), max(years) + 2), edgecolor="black", alpha=0.7)
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of Papers")
    ax.set_title("Papers by Publication Year")
    plt.tight_layout()
    plt.show()
else:
    print("No papers with year data.")

## Papers by Tag

In [ ]:
tag_counts = defaultdict(int)
for p in papers:
    for tag in p.tags:
        tag_counts[tag] += 1

if tag_counts:
    tags_sorted = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)
    labels, counts = zip(*tags_sorted)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels, counts, color="steelblue")
    ax.set_xlabel("Number of Papers")
    ax.set_title("Papers by Tag")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No tagged papers yet.")

## Top Cited Papers

In [ ]:
cited = sorted(
    [p for p in papers if p.citation_count],
    key=lambda p: p.citation_count,
    reverse=True,
)[:15]

if cited:
    df = pd.DataFrame([
        {"title": p.title[:60], "year": p.year, "citations": p.citation_count, "tags": ", ".join(p.tags)}
        for p in cited
    ])
    display(df)
else:
    print("No citation data available.")

## Tag Co-occurrence Network

Visualize which tags frequently appear together on the same paper.

In [ ]:
G = nx.Graph()
for p in papers:
    for i, tag_a in enumerate(p.tags):
        G.add_node(tag_a)
        for tag_b in p.tags[i+1:]:
            if G.has_edge(tag_a, tag_b):
                G[tag_a][tag_b]["weight"] += 1
            else:
                G.add_edge(tag_a, tag_b, weight=1)

if G.number_of_nodes() > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    nx.draw(
        G, pos, ax=ax,
        with_labels=True,
        node_size=1500,
        node_color="lightblue",
        font_size=9,
        width=[w * 0.5 for w in weights],
    )
    ax.set_title("Tag Co-occurrence Network")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough tagged papers to build a network.")